In [0]:
dbutils.fs.ls("abfss://bronze@grtadlsdev.dfs.core.windows.net")

In [0]:
df = spark.table("bronze.stop_times")
display(df.limit(50))

In [0]:
from pyspark.sql.functions import col

count_24 = df.filter(col("arrival_time").startswith("24:") | col("departure_time").startswith("24:")).count()
count_00 = df.filter(col("arrival_time").startswith("00:") | col("departure_time").startswith("00:")).count()

print("rows with 24:", count_24)
print("rows with 00:", count_00)

In [0]:
from pyspark.sql.functions import regexp_replace, col

df = df.withColumn("arrival_time", regexp_replace(col("arrival_time"), r"^24:", "00:")) \
       .withColumn("departure_time", regexp_replace(col("departure_time"), r"^24:", "00:"))

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.functions import to_timestamp, date_format, col

df = df.withColumn("arrival_time", date_format(to_timestamp(col("arrival_time"), "HH:mm:ss"), "HH:mm:ss")) \
       .withColumn("departure_time", date_format(to_timestamp(col("departure_time"), "HH:mm:ss"), "HH:mm:ss"))

In [0]:
%sql
create schema if not exists silver;

In [0]:
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.stop_times")

silver_path = "abfss://silver@grtadlsdev.dfs.core.windows.net/stop_times/"
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .save(silver_path)

In [0]:
from pyspark.sql.functions import col

# filter by exact time and stop id (use int or string depending on schema)
df.filter((col("arrival_time") == "19:00:00") & (col("stop_id") == 3489)).show(truncate=False)

# to get count instead of rows
count = df.filter((col("arrival_time") == "19:00:00") & (col("stop_id") == 3489)).count()
print(count)